# FrugalProver — budget labeling at scale with vLLM

Runs the oracle pipeline's first two stages end to end:

1. **Stage 1 `sample`** — draw a balanced problem set from MATH.
2. **Stage 2 `budget`** — label each problem with the solve effort it needs, by
   running the **solving agent** (`agent/` — prover → verifier → corrector) and
   recording the smallest budget that clears the success threshold (`b_star`).

Stage 2 is the expensive stage — it is the only one that generates text — so this
notebook is built around throughput.

## Why vLLM and not `HFClient`

`configs/agent/DeepSeek_R1_Distill_Qwen_7B.yaml` drives `transformers.generate`
directly. That path runs fixed-size waves of `max_batch_size` prompts, and the
whole wave waits for its slowest sequence. On a 4090 with 8-bit weights it
delivers roughly **90 output tokens/s**, which works out to ~3 problems/hour —
fine for a demo, hopeless for a few thousand problems.

`configs/agent/DeepSeek_R1_Distill_Qwen_7B_vllm.yaml` points the same loop at a
vLLM server through `agent/model.py:OpenAIClient`. vLLM batches *continuously*:
finished sequences leave the batch and queued ones join mid-flight, so the GPU
stays saturated. Same model, same prompts, same labels — one to two orders of
magnitude more throughput.

Two further savings are configured in that fragment:

- **`budget.single_pass_reconstruct`** — the three budgets used to mean three
  full re-solves. Because every budget is ≥ every role's per-call cap, a smaller
  budget produces the same calls in the same order and just stops sooner, so one
  pass at 32768 reconstructs all three exactly. ~2.2× less compute, and it
  couples the budgets to common random numbers, which removes resampling noise
  from `p(B₂) − p(B₁)`.
- **`agent.max_rounds: 3`** (from 4) — ~24% fewer tokens for one fewer repair.

## Sizing

Roughly `problems_in_5h = (server_tok/s × 18000 × 0.7) / (n_samples × tokens_per_attempt)`.
With `max_rounds: 3` and `n_samples: 3` that is ~37.5k output tokens per problem:

| GPU | ~output tok/s | problems in 5h |
|---|---|---|
| 1× 4090 / L4 (24GB) | 600 | ~200 |
| 1× A100 80GB | 1,500 | ~500 |
| 1× H100 | 3,000 | ~1,000 |
| 4× H100 (data-parallel) | 11,000 | ~3,700 |

These are estimates — **section 4 measures the real number** before you commit
the run. A single 24GB card tops out near 200 problems in 5 hours no matter how
the software is tuned; a few thousand needs multiple GPUs.

> **Runtime note:** a 16GB T4 has no bf16 and 15GB of fp16 weights leave no room
> for the KV cache. That runtime needs an AWQ/GPTQ checkpoint or a smaller model,
> not a flag change. A100, H100, L4 and 4090 are all fine at bf16.
>
> If you specifically want the local-`transformers` path instead (no server, e.g.
> for a quick single-problem trace), section 7 keeps it available.

## 1. Install

`.[openai]` is all the driver needs — it talks HTTP and imports no torch. `vllm`
is the server, installed here because it shares this runtime's GPU; on a split
setup it would only go on the GPU box.

`.[gpu]` is optional and only for section 7's local-`transformers` fallback.

In [ ]:
# Clone the repo (or run from an existing checkout).
!git clone https://github.com/newpotatato/smiles-frugalprover.git
%cd smiles-frugalprover
!git checkout "agent/DeepSeek_R1_Distill_Qwen_7B"

# [openai] = the OpenAI-compatible client the agent talks through. No torch: with
# the model on a server, the labeling driver is pure CPU code.
# vllm is the server itself. It pulls its own pinned torch, so install it in the
# same step and let pip resolve once.
!pip install -q -e ".[openai]"
!pip install -q vllm

# Optional: only needed for section 7 (local transformers, no server).
# !pip install -q -e ".[gpu]"

In [ ]:
# Confirm the GPU is visible and the package imports without a torch hoist.
!frugalprover info

## 2. Start the vLLM server

Launched in the background so the notebook keeps control. The first run also
downloads ~15GB of weights, so the health poll below is patient; if it times out,
the tail of the log says why.

**Set `N_GPUS` to the number of GPUs in this runtime.** Replicas are
*data*-parallel: a 7B fits one GPU with room for the KV cache, so tensor
parallelism would only add an all-reduce per layer to buy single-request latency,
which a batch job does not care about.

Three flags are correctness, not tuning:

- `--served-model-name` must equal `model:` in the config fragment, or every
  request 404s.
- `--max-model-len 24576` — measured, not estimated. The corrector prompt
  carries the candidate (≤8192) **and** the verifier's full critique via
  `flaws` (≤4096), plus problem and template: ~12.3k in. With 8192 out that is
  20481 — one token over a 20480 window — and vLLM 400s the moment the first
  repair round is reached, well into the run.
- **no `--kv-cache-dtype fp8`** — it halves KV memory and looks like free
  concurrency, but on this reasoning model it collapses generation into
  repetition loops (measured: 7/8 vs 0/8 for bf16). vLLM warns about it at
  startup; take the warning literally.
- **no `--reasoning-parser`** — it would move the chain-of-thought into
  `reasoning_content`, leaving `content` (what Stage 2 grades) with only the
  post-`</think>` text while `usage.completion_tokens` still counted the
  reasoning. The token axis and the text axis would measure different things.

In [ ]:
import os
import subprocess
import time

import requests

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
SERVED = "deepseek-r1-7b"          # must match `model:` in the vllm config fragment
PORT = 8000
BASE = f"http://127.0.0.1:{PORT}"
LOG = "/tmp/vllm.log"

N_GPUS = int(os.environ.get("N_GPUS", "1"))   # <-- set to the GPUs in this runtime

# Memory settings scale with the card, and the defaults are not safe everywhere.
# Measured on a 24GB 4090: --gpu-memory-utilization 0.92 budgets 21.6GiB, then
# needs 14.3 (bf16 weights) + ~7 (KV) + 0.9 (CUDA graphs) = ~22.5GiB and dies
# with CUDA OOM while capturing graphs. 0.88 leaves the graphs room.
#
# MAX_NUM_SEQS is a hard ceiling, not a wish: at fp8 this 7B costs ~28KB of KV
# per token (28 layers x 4 GQA KV heads x 128 dim x 2 for K+V), so ~7GiB of
# cache holds ~250k tokens -- about 25 concurrent 10k-token sequences. Asking
# for 256 on a 24GB card only makes vLLM preempt and recompute.
VRAM_GB = int(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split("\n")[0].strip()) // 1024
GMU, MAX_NUM_SEQS = ("0.88", 32) if VRAM_GB < 40 else ("0.92", 128)
print(f"{VRAM_GB}GB/GPU -> --gpu-memory-utilization {GMU}, --max-num-seqs {MAX_NUM_SEQS}")

cmd = [
    "vllm", "serve", MODEL,
    "--served-model-name", SERVED,
    "--dtype", "bfloat16",
    "--gpu-memory-utilization", GMU,
    "--max-model-len", "24576",
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--enable-prefix-caching",
    # No --kv-cache-dtype fp8. It reads as free throughput and it destroys
    # generation on this model -- measured A/B on a 4090, 8 problems, all else
    # fixed: fp8 gave 7/8 completions collapsing into repetition loops, each
    # running to the full 8192 cap; bf16 gave 0/8 degenerate, all stopping
    # naturally at 520-919 tokens. Degenerate output never stops early, so fp8
    # cost 11x the tokens and 20x the wall time for unusable text.
    "--port", str(PORT),
]
if N_GPUS > 1:
    cmd += ["--data-parallel-size", str(N_GPUS)]

# The agent reads its key from this variable (ModelSpec.api_key_env names the
# variable; it never holds the key inline).
#
# The SERVER inherits this environment too, and vLLM turns bearer auth ON for
# /v1/* whenever VLLM_API_KEY is set -- setting the variable is equivalent to
# passing --api-key. So every /v1 request needs the header below. Root-level
# routes (/health, /metrics, /tokenize) are exempt, which is why a missing header
# looks like a perfectly healthy server that 401s the moment you touch /v1.
API_KEY = "EMPTY"
os.environ["VLLM_API_KEY"] = API_KEY
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

log = open(LOG, "wb")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print(f"launched pid={server.pid}, logging to {LOG}\n  {' '.join(cmd)}\n")


def tail_log(n=40):
    return subprocess.run(["tail", f"-{n}", LOG], capture_output=True, text=True).stdout


deadline = time.time() + 1800        # generous: first run downloads the weights
while time.time() < deadline:
    # Checked every iteration: an OOM at startup kills the process in the first
    # minute, and without this the poll would wait the full 30 for a corpse.
    if server.poll() is not None:
        print(f"!! server exited with code {server.returncode}.")
        oom = [l for l in tail_log(400).splitlines() if "OutOfMemory" in l]
        if oom:
            print("\n>> CUDA OOM. Lower GMU (0.88 -> 0.85) and/or MAX_NUM_SEQS, then rerun.\n")
            print(oom[0][:400])
        print(tail_log(25))
        raise SystemExit(1)
    try:
        if requests.get(f"{BASE}/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print(tail_log())
    raise SystemExit("server did not become healthy in 30 min")

# /health is exempt from auth, so it passing proves only that the process is up.
# Check the status explicitly instead of indexing straight into the body: an auth
# or config failure returns valid JSON with no "data" key, and a blind ["data"]
# would surface that as an unrelated KeyError.
r = requests.get(f"{BASE}/v1/models", headers=HEADERS, timeout=10)
if r.status_code == 401:
    raise SystemExit(
        "401 from /v1/models -- the server has bearer auth on and the header did not "
        f"match. VLLM_API_KEY must be identical in this process and the server's "
        f"environment (this cell set both to {API_KEY!r}); if you started the server "
        "in a separate shell, export the same value there."
    )
if r.status_code != 200:
    print(tail_log(20))
    raise SystemExit(f"/v1/models returned {r.status_code}: {r.text[:500]}")

served = [m["id"] for m in r.json().get("data", [])]
print(f"server healthy. served models: {served}")
assert SERVED in served, f"--served-model-name mismatch: config wants {SERVED!r}, server has {served}"
print("OK: served name matches the config fragment, and /v1 auth works")

In [ ]:
# Sanity check the one response field the budget axis depends on.
#
# OpenAIClient counts tokens from `usage.completion_tokens`. That count is the
# ONLY input to the loop's budget cap, so if it were missing the client would
# fall back to a character heuristic and the cap would fire at the wrong place --
# with nothing in the output looking wrong. Confirm it is there before spending
# hours on the run.
resp = requests.post(
    f"{BASE}/v1/chat/completions",
    headers=HEADERS,          # /v1 is behind bearer auth -- see the cell above
    json={"model": SERVED,
          "messages": [{"role": "user", "content": "What is 2+2? Put the answer in \\boxed{}."}],
          "max_tokens": 64, "temperature": 0.6},
    timeout=120,
)
if resp.status_code != 200:
    raise SystemExit(f"/v1/chat/completions returned {resp.status_code}: {resp.text[:500]}")
r = resp.json()

print("content:", repr(r["choices"][0]["message"]["content"][:200]))
print("usage  :", r.get("usage"))
assert (r.get("usage") or {}).get("completion_tokens"), \
    "no usage.completion_tokens -- token accounting would be approximate"

# reasoning_content must be absent/empty. If the server was started with
# --reasoning-parser, the chain-of-thought moves there and `content` keeps only
# the post-</think> text, while completion_tokens still counts the reasoning --
# the token axis and the graded text would then measure different objects.
assert not (r["choices"][0]["message"].get("reasoning_content") or ""), \
    "server is splitting out reasoning_content -- restart it without --reasoning-parser"

print("\nOK: exact token accounting, and content holds the full generation")

## 3. Stage 1 — sample problems from MATH

`frugalprover sample` draws a balanced set and writes `data/<run>/problems.jsonl`
(A1 records). All later stages resolve files inside the same run dir, so keep
`--run-name` consistent.

Draw **more problems than you expect to label**. Stage 2 stops on its time
budget, and `budget.shuffle` makes whatever fraction it reaches a representative
subsample — so oversampling costs nothing and undersampling caps the run.

`sample` is cheap and needs no GPU (it only reads the dataset), so the full
balanced draw across all seven subjects is the default here.

In [ ]:
RUN = "label5h"

# Full balanced draw: all 7 MATH subjects x 5 levels. per_level_per_subject=200
# gives a pool of up to 7000, capped to n_problems. Shrink n_problems if you know
# the runtime is small -- but note Stage 2 already stops on its own deadline.
!frugalprover sample -c configs/base.yaml --run-name {RUN} \
  --set sample.per_level_per_subject=200 \
  --set sample.n_problems=5000

# For a quick demo instead, use a handful:
# !frugalprover sample -c configs/base.yaml --run-name demo \
#   --set "sample.subjects=[algebra, number_theory]" \
#   --set sample.per_level_per_subject=2 --set sample.n_problems=6

## 4. Pilot — measure before committing the run

**The highest-value fifteen minutes in this notebook.** Two things can quietly
waste the whole window, and both are visible on 20 problems.

**4a — verifier acceptance.** `parse_critique` fails safe to REJECT when it
can't find a `VERDICT:` line. R1-Distill buries its output behind a long
`<think>` block, so if the verdict gets truncated — or is emitted as
`**VERDICT:** ACCEPT`, which the current `startswith` check does not match — the
loop *never* accepts and burns all `max_rounds` on every attempt. That roughly
doubles the cost of the run and makes the labels reflect a broken critic.
Healthy is **20–50% first-round acceptance**.

`prove` calls `solve_batch(max_new_tokens=0, ...)`, so the cap is disabled — this
measures per-role cost and acceptance with no truncation interference.

**4b — real throughput.** A 50-problem `budget` run gives seconds/problem, which
sets `max_problems` for the real run.

In [ ]:
# 4a -- acceptance rate and per-role token cost, 20 problems, cap disabled.
VLLM_CFG = "configs/agent/DeepSeek_R1_Distill_Qwen_7B_vllm.yaml"

!frugalprover prove -c configs/base.yaml -c {VLLM_CFG} \
  --problems data/{RUN}/problems.jsonl --max-problems 20 \
  --run-name pilot --out prove_pilot.jsonl

In [ ]:
# Read the pilot: acceptance rate, rounds histogram, tokens per attempt.
import collections
import statistics

from frugalprover.common.io import read_jsonl

rows = list(read_jsonl("data/pilot/prove_pilot.jsonl"))
acc = sum(bool(r.get("accepted")) for r in rows) / len(rows)
rounds = collections.Counter(r["rounds"] for r in rows)
toks = [r["tokens"] for r in rows]

print(f"acceptance rate : {acc:.0%}   (healthy: 20-50%)")
print(f"rounds histogram: {dict(sorted(rounds.items()))}")
print(f"tokens/attempt  : mean {statistics.mean(toks):.0f}, median {statistics.median(toks):.0f}")

MAX_ROUNDS = 3
if acc < 0.05 and rounds.get(MAX_ROUNDS, 0) > 0.8 * len(rows):
    print(
        "\n!! NEVER-ACCEPTS MODE. Every attempt is running the full round budget,\n"
        "   which roughly doubles the cost of the run. Diagnose before proceeding:\n"
        "     1. Truncation? Re-run with --set agent.verifiers[0].max_tokens=8192.\n"
        "        If acceptance jumps, prepend 'Output the VERDICT line FIRST,\n"
        "        before any reasoning.' to VERIFY_PROMPT in common/config.py (free).\n"
        "     2. Format? Inspect raw verifier text below. If it says '**VERDICT:**\n"
        "        ACCEPT' or '### Verdict: ACCEPT', roles.py:126 startswith() is\n"
        "        missing it -- replace with a regex over the whole completion:\n"
        "           re.search(r'VERDICT\\W{0,4}(ACCEPT|REJECT)', raw, re.I)  # last match\n"
        "     3. Genuinely harsh critic? Soften the verifier prompt's framing."
    )
else:
    print("\nOK: the verify-repair loop is converging.")

print("\n--- flaws cited on a rejected attempt (sanity-check the critic) ---")
for r in rows:
    if r.get("flaws"):
        print(r["flaws"][:3])
        break

In [ ]:
# 4b -- real Stage 2 on 50 problems, in two batches, to time it and prove resume.
import json
import time

t0 = time.time()
!frugalprover budget -c configs/base.yaml -c {VLLM_CFG} --run-name pilot \
  --set budget.problems={RUN}/problems.jsonl \
  --set budget.max_problems=50 --set budget.batch_size=25 \
  --set budget.time_budget_s=2700
pilot_s = time.time() - t0

meta = json.load(open("data/pilot/budgets.jsonl.meta.json"))
recs = [json.loads(l) for l in open("data/pilot/budgets.jsonl")]
n = meta["n_labeled_this_run"] or len(recs)
per_problem = pilot_s / max(1, n)
gen = [r.get("tokens_generated") for r in recs if r.get("tokens_generated")]

print(f"\nb_star distribution : {meta['b_star_distribution']}")
print(f"censored            : {meta['n_censored']}/{meta['n_records']}")
print(f"reconstructed       : {sum(bool(r.get('reconstructed')) for r in recs)}/{len(recs)}")
print(f"seconds/problem     : {per_problem:.1f}")
if gen:
    print(f"tokens generated/problem: {statistics.mean(gen):.0f}")

FIVE_HOURS = 5 * 3600
projected = int(FIVE_HOURS / per_problem)
print(f"\n==> projected in 5h : ~{projected} problems")
print(f"==> set max_problems: {int(0.85 * projected)}  (15% slack on the estimate)")

# Go/no-go on the label distribution itself.
d = meta["b_star_distribution"]
if meta["n_censored"] == meta["n_records"]:
    print("\n!! everything censored -- budgets too small, or the loop is broken (see 4a)")
elif meta["single_pass"] or sum(1 for k, v in d.items() if k != "censored" and v) < 2:
    print("\n!! b_star lands in one bucket -- the budget axis carries no signal.")
    print("   Shift `budgets` so they bracket the observed tokens/problem above.")
else:
    print("\nOK: b_star spreads across budgets -- the axis is informative.")
if not gen:
    print("\n!! no tokens_generated -- reconstruction fell back. Check the log for the")
    print("   'falling back to N independent passes' warning; the run will cost ~2.2x more.")

## 5. Stage 2 — the production labeling run

Stops on its own after `time_budget_s` (4h50m by default), finishing the batch
in flight first. It is **resumable**: rerun the same command after a disconnect
and it labels only what's missing.

Two settings make a truncated run scientifically usable rather than merely
partial:

- **`budget.shuffle`** — `problems.jsonl` is sorted by id, i.e. grouped by
  subject. Without the shuffle a run cut short would label all of `algebra` and
  none of `precalculus`, and `subject` is a feature in the surface baseline. With
  it, whatever fraction completes is a representative subsample.
- **`budget.batch_size: 64`** — 64 × 3 samples = 192 requests in flight, enough
  to keep the replicas fed. Also the checkpoint granularity, since records reach
  disk only after a whole batch completes.

Set `max_problems` from the pilot projection above. Run the monitor cell (6) in
parallel to watch throughput.

In [ ]:
MAX_PROBLEMS = int(0.85 * projected)   # from the pilot; override by hand if you prefer

# time_budget_s, shuffle, single_pass_reconstruct, batch_size and
# continue_on_error all come from the vllm fragment -- see it for the reasoning.
!frugalprover budget -c configs/base.yaml -c {VLLM_CFG} --run-name {RUN} \
  --set budget.max_problems={MAX_PROBLEMS}

## 6. Monitor — is the GPU actually busy?

Run this from a second cell (or after a partial run) while labeling is going.

The number that matters is `num_requests_running` against `--max-num-seqs`. If
it sits well below, the GPU is idle waiting on the loop, not the other way
round: the verify-repair loop advances in lockstep, so every attempt waits for
the slowest one in its round, and the tail of a round starves the server. The
fix is a larger `budget.batch_size`, not a server flag.

In [ ]:
import requests

# /metrics is a root-level route, so it is exempt from the /v1 bearer auth --
# the header is sent anyway so this cell works against a gateway that guards
# everything.
resp = requests.get(f"{BASE}/metrics", headers=HEADERS, timeout=10)
if resp.status_code != 200:
    raise SystemExit(f"/metrics returned {resp.status_code}: {resp.text[:300]}")

want = ("num_requests_running", "num_requests_waiting",
        "generation_tokens_total", "prompt_tokens_total",
        "gpu_cache_usage_perc", "gpu_prefix_cache_hit_rate")
vals = {}
for line in resp.text.splitlines():
    if line.startswith("#"):
        continue
    for w in want:
        if f"vllm:{w}" in line:
            try:
                vals[w] = vals.get(w, 0.0) + float(line.rsplit(" ", 1)[1])
            except (ValueError, IndexError):
                pass

for w in want:
    if w in vals:
        print(f"{w:28s} {vals[w]:,.2f}")

running = vals.get("num_requests_running", 0)
MAX_NUM_SEQS = globals().get("MAX_NUM_SEQS", 32)  # set by the launch cell
print()
if running < 0.25 * MAX_NUM_SEQS:
    print(f"!! only {running:.0f}/{MAX_NUM_SEQS} sequences running -- the GPU is starved.")
    print("   The loop advances in lockstep, so the tail of each round leaves the")
    print("   server idle. That is a client-side bottleneck: raise budget.batch_size")
    print("   (currently 64) rather than changing a server flag.")
else:
    print(f"OK: {running:.0f}/{MAX_NUM_SEQS} sequences in flight -- server is the bottleneck.")

## 7. Inspect the budget labels

`b_star` is the smallest budget that cleared the threshold (`null` = censored,
not solved within any budget). `p` is the success rate per budget, `sc` the
self-consistency (plurality-vote) baseline the oracle has to beat, `tokens_spent`
the sweep-shaped total. Under reconstruction each record also carries
`tokens_generated` -- what the single pass actually cost -- and
`reconstructed: true`.


In [ ]:
import collections
import json

from frugalprover.common.config import load_config
from frugalprover.common.io import read_jsonl

cfg = load_config(["configs/base.yaml", VLLM_CFG], overrides=[f"run_name={RUN}"])
rows = list(read_jsonl(cfg.data_path("budgets.jsonl")))

for row in rows[:10]:
    print(f"[{row['id']}] b_star={row['b_star']} p={row['p']} "
          f"sc={row['sc']} tokens_spent={row['tokens_spent']}")
print(f"... {len(rows)} records total\n")

meta = json.load(open(cfg.data_path("budgets.jsonl.meta.json")))
print("b_star distribution:", meta["b_star_distribution"])
print("stopped early      :", meta["stopped_early"],
      "| remaining:", meta["n_remaining"],
      "| elapsed:", f"{meta['elapsed_s'] / 3600:.2f}h")

# The shuffle's payoff: coverage should be spread across subjects and levels even
# if the run was cut short. A truncated UNshuffled run would show one or two
# subjects only -- and `subject` is a feature in the surface baseline.
probs = {p["id"]: p for p in read_jsonl(cfg.data_path("problems.jsonl"))}
subj = collections.Counter(probs[r["id"]]["type"] for r in rows if r["id"] in probs)
lvl = collections.Counter(probs[r["id"]]["level"] for r in rows if r["id"] in probs)
print("\nlabeled by subject:", dict(sorted(subj.items())))
print("labeled by level  :", dict(sorted(lvl.items(), key=lambda kv: (kv[0] is None, kv[0]))))

## 8. Full trace -- watch the agent solve one problem step by step

The labels above are a summary. This cell drives the same building blocks
`VerifyRepairAgent` uses internally (`Prover`, `Verifier`, `Corrector`,
`aggregate`, `collect_flaws`) for a single problem, printing every round instead
of batching silently: the prover's candidate, each verifier's verdict and
diagnosed flaws, the corrector's repair, and the final grading.

This is also the fastest way to *see* the acceptance problem section 4a screens
for -- if the verifier's raw text says `**VERDICT:** ACCEPT` but the parsed
verdict prints REJECT, that is the format bug.

Goes through `build_model_client`, so it uses whichever client the loaded config
names -- the vLLM server here, with no change.


In [ ]:
import textwrap

from frugalprover.agent.aggregation import aggregate, collect_flaws
from frugalprover.agent.model import build_model_client
from frugalprover.agent.roles import Corrector, Prover, Verifier
from frugalprover.common.grading import grade
from frugalprover.common.records import ProblemRecord


def _show(label: str, text: str) -> None:
    print(f"--- {label} ---")
    print(textwrap.shorten(text, width=2000, placeholder=" [...]"))
    print()


problem = ProblemRecord.from_dict(next(read_jsonl(cfg.data_path("problems.jsonl"))))
print(f"Problem [{problem.id}]: {problem.problem}\nGold answer: {problem.answer}\n")

a = cfg.agent
prover = Prover(build_model_client(a.prover), a.prover, a.prover_prompt, "prover")
corrector = Corrector(build_model_client(a.corrector), a.corrector, a.corrector_prompt, "corrector")
verifiers = [Verifier(build_model_client(v), v, a.verifier_prompt, "verifier") for v in a.verifiers]
for role in [prover, corrector, *verifiers]:
    role.client.setup()

try:
    candidate = prover.propose([problem.problem])[0]
    tokens = prover.count_tokens(candidate)
    _show("round 0 -- prover proposes", candidate)

    accepted = False
    for round_i in range(a.max_rounds):
        critiques = [v.audit([(problem.problem, candidate)])[0] for v in verifiers]
        tokens += sum(v.count_tokens(c.raw) for v, c in zip(verifiers, critiques))
        for i, c in enumerate(critiques):
            _show(f"round {round_i + 1} -- verifier {i} ({'ACCEPT' if c.accept else 'REJECT'})",
                  c.raw or "(empty response)")

        if aggregate(critiques, a.aggregation):
            accepted = True
            print(f"Converged after round {round_i + 1} ({a.aggregation} rule).\n")
            break

        flaws = collect_flaws(critiques)
        candidate = corrector.repair([(problem.problem, candidate, flaws)])[0]
        tokens += corrector.count_tokens(candidate)
        _show(f"round {round_i + 1} -- corrector repairs", candidate)
    else:
        status = "flagged" if a.on_nonconvergence == "flag" else "rejected"
        print(f"Hit max_rounds={a.max_rounds} without converging -> {status}.\n")

    correct = grade(candidate, problem.answer)
    print(f"Final candidate {'PASSES' if correct else 'FAILS'} grading against the gold answer "
          f"(accepted={accepted}); total tokens spent: {tokens}")
finally:
    for role in [prover, corrector, *verifiers]:
        role.client.teardown()

## 9. (Optional) the local-`transformers` path

`HFClient` runs the model in-process with no server — slow for a full labeling
run, but useful as a comparison point or when you can't run a server. Needs
`pip install -e ".[gpu]"` from section 1.

Note the two paths count tokens differently: `HFClient` decodes with
`skip_special_tokens=True` and re-encodes the result, while the vLLM path uses
the server's `usage.completion_tokens`. The difference is a few tokens per call,
but it means a corpus labeled half one way and half the other is not one
labeling run. Both are recorded in the A2 sidecar via `describe()`.

In [ ]:
from frugalprover.agent.model import build_model_client
from frugalprover.common.config import ModelSpec

# client="openai" would reuse the running server instead; "hf" loads weights here.
client = build_model_client(ModelSpec(
    client="hf", model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
))
client.setup()
out = client.generate(
    ["Problem: Compute 17 * 23. Give the answer in \\boxed{}."],
    max_tokens=512, temperature=0.6, top_p=0.95, role="prover",
)
print(out[0])
client.teardown()

## 10. Shut down the server

Frees the GPU. Worth running explicitly — a backgrounded `vllm serve` survives a
kernel restart and will hold the whole card, so the next launch fails on memory
with a message that doesn't obviously point back here.

Labeling is resumable, so stopping the server does not lose work: rerun section 5
after restarting it and the run picks up from `budgets.jsonl`.

In [ ]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()
    server.wait(timeout=30)
log.close()
print(f"server stopped (exit {server.returncode})")

# Catch anything left over from an earlier kernel that this handle doesn't own.
!pkill -f "vllm serve" 2>/dev/null; nvidia-smi --query-gpu=index,memory.used --format=csv